In [3]:
import duckdb
import pandas as pd

# Combining the bronze layer tables
From the bronze layer, there are 4 separate tables, one for each of the indicators. The first step I will make is to join all 4 tables together, using the date as a joining key.

In [44]:
import duckdb
con = duckdb.connect("econ.duckdb")
total_metrics_df = con.sql("""SELECT 
    u.date AS quarter, 
    u.value AS unemployment_rate, 
    i.value AS inflation_rate, 
    g.value AS gdp_in_gbp_m, 
    t.value AS trade_balance_gbp_m
    FROM unemployment AS u
    JOIN inflation AS i
    ON i.date = u.date
    JOIN GDP AS g
    ON g.date = u.date
    JOIN trade_balance AS t
    ON t.date = u.date
    ORDER BY u.date""").df()

# Cleaning the tables
After joining the tables, I will clean the data by using the columns - which were all in VARCHAR format, to their rightful data formats - also by adding columns for the year, quarter and the date to make analysis over time easier. 

In [46]:
total_metrics_df = con.sql("""SELECT quarter AS quarter_raw,
    CAST(SUBSTRING(quarter,1,4) AS INTEGER) AS year,
    CAST(SUBSTRING(quarter,7,1) AS INTEGER) AS quarter_number,
    MAKE_DATE(
        CAST(SUBSTRING(quarter,1,4) AS INTEGER),
        (CAST(SUBSTRING(quarter,7,1) AS INTEGER) - 1) * 3 + 1,
        1
    ) AS date,
    CAST(NULLIF(unemployment_rate, '') AS DOUBLE) AS unemployment_rate,
    CAST(NULLIF(inflation_rate, '') AS DOUBLE) AS inflation_rate,
    CAST(NULLIF(gdp_in_gbp_m, '') AS DOUBLE) AS gdp_in_gbp_m,
    CAST(NULLIF(trade_balance_gbp_m, '') AS DOUBLE) AS trade_balance_gbp_m
FROM total_metrics_df""")



In [47]:
display(total_metrics_df)

┌─────────────┬───────┬────────────────┬────────────┬───────────────────┬────────────────┬──────────────┬─────────────────────┐
│ quarter_raw │ year  │ quarter_number │    date    │ unemployment_rate │ inflation_rate │ gdp_in_gbp_m │ trade_balance_gbp_m │
│   varchar   │ int32 │     int32      │    date    │      double       │     double     │    double    │       double        │
├─────────────┼───────┼────────────────┼────────────┼───────────────────┼────────────────┼──────────────┼─────────────────────┤
│ 1989 Q1     │  1989 │              1 │ 1989-01-01 │               7.6 │            5.0 │     374612.0 │                NULL │
│ 1989 Q2     │  1989 │              2 │ 1989-04-01 │               7.2 │            5.3 │     376951.0 │                NULL │
│ 1989 Q3     │  1989 │              3 │ 1989-07-01 │               7.1 │            5.1 │     377573.0 │                NULL │
│ 1989 Q4     │  1989 │              4 │ 1989-10-01 │               7.0 │            5.5 │     377508.0 

# Removing rows
Lastly, I made the decision to keep the rows from 1997 and onwards. This was because there were rows containing no data for trade balance for 1996 and below.

In [ ]:
con.execute("CREATE OR REPLACE TABLE silver_quarterly_metrics AS SELECT * FROM total_metrics_df WHERE date > '1996-10-1'")

NameError: name 'con' is not defined

In [63]:
con.sql("DESCRIBE silver_quarterly_metrics")

┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ quarter_raw         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ year                │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ quarter_number      │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ date                │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ unemployment_rate   │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ inflation_rate      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ gdp_in_gbp_m        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ trade_balance_gbp_m │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────────────┴─────────────┴─────────┴─────────┴───────

In [72]:
con.close()